# Setup Environment

## Only Run on First Set-Up

In [1]:
"""
%%bash
set -eo pipefail
ENV_NAME="rag-capstone" PLATFORM="linux-64" bash scripts/bootstrap_env.sh


mkdir -p "$CONDA_PREFIX/etc/conda/activate.d"
cat > "$CONDA_PREFIX/etc/conda/activate.d/java.sh" <<'EOF'

set -euo pipefail

export JAVA_HOME="${CONDA_PREFIX}"
export PATH="${CONDA_PREFIX}/bin:${PATH}"
EOF
chmod +x "$CONDA_PREFIX/etc/conda/activate.d/java.sh"
"""

'\n%%bash\nset -eo pipefail\nENV_NAME="rag-capstone" PLATFORM="linux-64" bash scripts/bootstrap_env.sh\n\n\nmkdir -p "$CONDA_PREFIX/etc/conda/activate.d"\ncat > "$CONDA_PREFIX/etc/conda/activate.d/java.sh" <<\'EOF\'\n\nset -euo pipefail\n\nexport JAVA_HOME="${CONDA_PREFIX}"\nexport PATH="${CONDA_PREFIX}/bin:${PATH}"\nEOF\nchmod +x "$CONDA_PREFIX/etc/conda/activate.d/java.sh"\n'

## Run each kernal restart

In [2]:
%%bash
source "$(conda info --base)/etc/profile.d/conda.sh"
conda activate rag-capstone

In [3]:
import os
import subprocess
from pathlib import Path

def _first_match(root: Path, name: str) -> str:
    for p in root.rglob(name):
        if p.is_file():
            return str(p)
    raise RuntimeError(f"Could not find {name} under {root}")

env_prefix = str(Path(os.sys.executable).resolve().parent.parent)

javac = _first_match(Path(env_prefix), "javac")
libjvm = _first_match(Path(env_prefix), "libjvm.so")

java_home = str(Path(libjvm).resolve().parent.parent.parent)

os.environ["CONDA_PREFIX"] = env_prefix
os.environ["JAVA_HOME"] = java_home
os.environ["JVM_PATH"] = libjvm
os.environ["PATH"] = f"{env_prefix}/bin:{java_home}/bin:" + os.environ.get("PATH", "")

print("Python:", os.sys.executable)
print("CONDA_PREFIX:", os.environ["CONDA_PREFIX"])
print("JAVA_HOME:", os.environ["JAVA_HOME"])
print("JVM_PATH:", os.environ["JVM_PATH"])
print("javac:", subprocess.check_output(["which", "javac"]).decode().strip())
print(subprocess.check_output(["javac", "-version"], stderr=subprocess.STDOUT).decode().strip())

Python: /home/sagemaker-user/.conda/envs/rag-capstone/bin/python
CONDA_PREFIX: /home/sagemaker-user/.conda/envs/rag-capstone
JAVA_HOME: /home/sagemaker-user/.conda/envs/rag-capstone/lib/jvm
JVM_PATH: /home/sagemaker-user/.conda/envs/rag-capstone/lib/jvm/lib/server/libjvm.so
javac: /home/sagemaker-user/.conda/envs/rag-capstone/bin/javac
javac 21.0.10-internal


In [4]:
%%capture
from src.utils.aws_secrets import bootstrap_env
bootstrap_env()

# Run Experiment

## Imports and Settings

In [5]:
import os
import logging
import json
import pickle
import litellm
import boto3

from datetime import datetime
from pathlib import Path

from src.utils.config import PipelineConfig
from src.pipeline import run_experiment
from src.utils.helpers import format_results_dataframe

/home/sagemaker-user/.conda/envs/rag-capstone/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/sagemaker-user/.conda/envs/rag-capstone/lib/python3.11/site-packages/sympy/external/gmpy.py:138: UserWarning: gmpy2 version is too old to use (2.0.0 or newer required)
  gmpy = import_module('gmpy2', min_module_version=_GMPY2_MIN_VERSION,
2026-03-15 05:18:38.525228084 [W:onnxruntime:Default, device_discovery.cc:211 DiscoverDevicesForPlatform] GPU device discovery failed: device_discovery.cc:91 ReadFileContents Failed to open file: "/sys/class/drm/card0/device/vendor"


## S3 Bucket Access Check

In [6]:
import boto3, os

s3      = boto3.client("s3")
bucket  = "sagemaker-us-east-1-168400405789"
indexes = "pyserini-indexes"
outputs = "outputs/jane"  # change to your name

tests = [
    ("LIST  indexes", lambda: s3.list_objects_v2(Bucket=bucket, Prefix=f"{indexes}/", MaxKeys=1)),
    ("WRITE outputs", lambda: s3.put_object(Bucket=bucket, Key=f"{outputs}/test.txt", Body=b"test")),
    ("READ  outputs", lambda: s3.get_object(Bucket=bucket, Key=f"{outputs}/test.txt")),
    ("DELETE test  ", lambda: s3.delete_object(Bucket=bucket, Key=f"{outputs}/test.txt")),
]

for name, fn in tests:
    try:    fn(); print(f"✓ {name}")
    except Exception as e: print(f"✗ {name} — {e}")

✓ LIST  indexes
✓ WRITE outputs
✓ READ  outputs
✓ DELETE test  


In [7]:
os.environ["LITELLM_LOG"] = "ERROR"
logging.getLogger("LiteLLM").setLevel(logging.ERROR)
logging.getLogger("litellm").setLevel(logging.ERROR)

#S3 bucket for shared indexes and outputs
os.environ["S3_BUCKET"]         = "sagemaker-us-east-1-168400405789"
os.environ["S3_INDEX_PREFIX"]   = "pyserini-indexes"
os.environ["S3_OUTPUT_PREFIX"]  = "outputs/jane"  ##change to your name

## Load Pyserini Indexes from S3

In [8]:
# ── Copy Pyserini indexes from S3 to /tmp ─────────────────────────────────────
# Runs on every kernel restart (indexes are lost when instance stops).

s3          = boto3.client("s3")
s3_bucket   = os.environ["S3_BUCKET"]
s3_prefix   = os.environ["S3_INDEX_PREFIX"]
local_cache = os.environ["PYSERINI_CACHE"]   # /tmp/pyserini_cache

if os.path.exists(local_cache) and any(Path(local_cache).rglob("*")):
    print(f"✓ Pyserini indexes already in {local_cache} — skipping download.")
else:
    paginator = s3.get_paginator("list_objects_v2")
    pages     = paginator.paginate(Bucket=s3_bucket, Prefix=s3_prefix + "/")
    objects   = [obj for page in pages for obj in page.get("Contents", [])]

    if not objects:
        print("No index files found in S3 — Pyserini will download from upstream on first use.")
    else:
        print(f"Downloading {len(objects)} index file(s) from "
              f"s3://{s3_bucket}/{s3_prefix}/ → {local_cache} ...")
        for obj in objects:
            s3_key = obj["Key"]
            rel    = s3_key[len(s3_prefix) + 1:]   # strip prefix + leading /
            if not rel:
                continue
            dst = os.path.join(local_cache, rel)
            os.makedirs(os.path.dirname(dst), exist_ok=True)
            s3.download_file(s3_bucket, s3_key, dst)
        print(f"✓ Done — indexes cached in {local_cache}.")


✓ Done — indexes cached in /tmp/pyserini_cache.


In [9]:
# ── S3 output helpers ─────────────────────────────────────────────────────────

def _s3_upload(local_path: str, s3_key: str) -> None:
    """Upload a local file to the shared S3 output bucket."""
    s3_client = boto3.client("s3")
    bucket    = os.environ["S3_BUCKET"]
    s3_client.upload_file(local_path, bucket, s3_key)
    print(f"  ✓ Uploaded → s3://{bucket}/{s3_key}")


def _s3_output_key(filename: str) -> str:
    """Build an S3 key under the configured output prefix."""
    prefix = os.environ["S3_OUTPUT_PREFIX"].rstrip("/")
    return f"{prefix}/{filename}"


## Load Data

In [10]:
queries_path = os.path.join(os.getcwd(), "hotpotqa", "hotpot_eval_300.json")
with open(queries_path, 'r', encoding='utf-8') as file:
        queries = json.load(file)[:1]

## Config

In [11]:

os.environ["S3_BUCKET"] = "sagemaker-us-east-1-168400405789"
os.environ["S3_INDEX_PREFIX"] = "pyserini-indexes"
os.environ["S3_OUTPUT_PREFIX"] = "outputs/jane"

# Tell Pyserini where to look for local cache (which we will sync from S3)
os.environ["PYSERINI_CACHE"] = os.path.join(os.getcwd(), "pyserini_cache")

In [12]:
baseline_cfg = PipelineConfig(
    iterative=True,
    embedding_type="fused",
    top_k=25,
    k_sparse=50,
    k_dense=50,
    rrf_k=60,
    use_rerank=True,
    top_n=5,
    max_length=512,
    batch_size=32,
    temperature=0.0,
    max_tries=4,
    eval_temperature=0.0,
    eval_max_tokens=2048,
    log_full_prompts=False,
    max_rounds=3,
    max_plan_steps=6,
    max_contexts_final=None,
    step_top_k=5,
    use_mlflow=True,
    mlflow_artifact_dir="artifacts",
)

## Run and Save Experiment

In [13]:
raw_results = run_experiment(queries=queries, cfg=baseline_cfg)

timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

# ── Local temp paths ──────────────────────────────────────────────────────────
local_tmp = "/tmp/results"
os.makedirs(local_tmp, exist_ok=True)

raw_results_local       = os.path.join(local_tmp, f"raw_results_{timestamp}.pkl")
formatted_results_local = os.path.join(local_tmp, f"formatted_results_{timestamp}.pkl")

# ── Save locally first ────────────────────────────────────────────────────────
with open(raw_results_local, "wb") as f:
    pickle.dump(raw_results, f)

formatted_results = format_results_dataframe(examples=queries, results=raw_results)
with open(formatted_results_local, "wb") as f:
    pickle.dump(formatted_results, f)

# ── Upload both to S3 ─────────────────────────────────────────────────────────
print("Uploading results to S3 ...")
_s3_upload(raw_results_local,       _s3_output_key(f"raw_results_{timestamp}.pkl"))
_s3_upload(formatted_results_local, _s3_output_key(f"formatted_results_{timestamp}.pkl"))
print("Done.")
print(f"  S3 prefix: s3://{os.environ['S3_BUCKET']}/{os.environ['S3_OUTPUT_PREFIX']}/")


  0%|          | 0/1 [00:00<?, ?it/s]

Attempting to initialize prebuilt index beir-v1.0.0-hotpotqa.splade-v3.
/home/sagemaker-user/mids-capstone-iterative-rag-critic/pyserini_cache/indexes/lucene-inverted.beir-v1.0.0-hotpotqa.splade-v3.20250603.168a2d.55d5635f4af0979d880daa3e8a361440 already exists, skipping download.
Initializing beir-v1.0.0-hotpotqa.splade-v3...


Mar 15, 2026 5:20:07 AM org.apache.lucene.store.MemorySegmentIndexInputProvider <init>
INFO: Using MemorySegmentIndexInput with Java 21; to disable start with -Dorg.apache.lucene.store.MMapDirectory.enableMemorySegments=false


Attempting to initialize prebuilt index beir-v1.0.0-hotpotqa.bge-base-en-v1.5.
/home/sagemaker-user/mids-capstone-iterative-rag-critic/pyserini_cache/indexes/faiss-flat.beir-v1.0.0-hotpotqa.bge-base-en-v1.5.20240107.d2c08665e8cd750bd06ceb7d23897c94 already exists, skipping download.
Initializing beir-v1.0.0-hotpotqa.bge-base-en-v1.5...



2026-03-15 05:23:17,748 - asyncio - ERROR - Task was destroyed but it is pending!
task: <Task pending name='Task-94' coro=<LoggingWorker._worker_loop() running at /home/sagemaker-user/.conda/envs/rag-capstone/lib/python3.11/site-packages/litellm/litellm_core_utils/logging_worker.py:121> wait_for=<Future finished result=None>>
/home/sagemaker-user/.conda/envs/rag-capstone/lib/python3.11/site-packages/litellm/litellm_core_utils/logging_worker.py:77: RuntimeWarning: coroutine 'Logging.async_success_handler' was never awaited
  self._worker_task = None
100%|██████████| 1/1 [03:21<00:00, 201.88s/it, last_s=201.87]


Uploading results to S3 ...
  ✓ Uploaded → s3://sagemaker-us-east-1-168400405789/outputs/jane/raw_results_2026-03-15_05-23-26.pkl
  ✓ Uploaded → s3://sagemaker-us-east-1-168400405789/outputs/jane/formatted_results_2026-03-15_05-23-26.pkl
Done.
  S3 prefix: s3://sagemaker-us-east-1-168400405789/outputs/jane/


In [16]:
import pickle, boto3, pandas as pd
from io import BytesIO

s3     = boto3.client("s3")
bucket = os.environ["S3_BUCKET"]
prefix = os.environ["S3_OUTPUT_PREFIX"]

for obj in s3.list_objects_v2(Bucket=bucket, Prefix=prefix+"/").get("Contents", []):
    if not obj["Key"].endswith(".pkl"):
        continue
    data    = pickle.loads(s3.get_object(Bucket=bucket, Key=obj["Key"])["Body"].read())
    df      = data if isinstance(data, pd.DataFrame) else pd.json_normalize(data)
    csv_key = obj["Key"].replace(".pkl", ".csv")
    buf     = BytesIO(); df.to_csv(buf, index=False); buf.seek(0)
    s3.put_object(Bucket=bucket, Key=csv_key, Body=buf.getvalue())
    print(f"✓ {csv_key}")

✓ outputs/jane/formatted_results_2026-03-15_05-23-26.csv
✓ outputs/jane/raw_results_2026-03-15_05-23-26.csv
